# NumPy in Scientific Toxicology
### Arrays · Broadcasting · Linear Algebra · Statistics · Dose-Response · Fingerprints

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

NumPy is the computational backbone of every scientific Python workflow. This notebook teaches NumPy **through problems you actually face** in toxicology and cheminformatics.

| Section | Topics |
|---------|--------|
| 1. Arrays & dtypes | creation, shapes, memory layout |
| 2. Indexing & slicing | Boolean masks, fancy indexing |
| 3. Broadcasting | Descriptor normalisation, similarity |
| 4. Linear algebra | PCA, regression, matrix operations |
| 5. Statistics | Descriptive stats, hypothesis testing |
| 6. Dose-response | Hill equation, EC50, LD50 |
| 7. Fingerprints | Tanimoto, cosine, Dice at scale |
| 8. Signal processing | FFT, smoothing for MEA/assay data |
| 9. Random & simulation | Monte Carlo, bootstrapping |
| 10. Performance tips | Vectorisation, memory, profiling |

---
## Section 1 — Arrays and Dtypes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED

print(f"NumPy version: {np.__version__}")

# ── 1.1 Array creation ────────────────────────────────────────────────────────
# Concentrations in a dose-response assay (μM)
concs = np.array([0.001, 0.01, 0.1, 1.0, 10.0, 100.0])
print("Concentrations (μM):", concs)

# Log-space concentrations (more natural for dose-response)
concs_log = np.logspace(-3, 2, 7)   # 10^-3 to 10^2, 7 points
print("Log-spaced (μM):    ", concs_log.round(4))

# Dose grid for an 8-point dilution series (2-fold)
top_dose = 100.0
doses = top_dose / (2 ** np.arange(8))
print("2-fold dilution:    ", doses)

# Zeros, ones, identity — used in linear algebra
blank_plate  = np.zeros((8, 12))     # 96-well plate values
vehicle_ctrl = np.ones(24) * 100.0   # 100% viability
print(f"\n96-well plate shape: {blank_plate.shape}")
print(f"Vehicle controls: {vehicle_ctrl[:5]}... (% viability)")

In [ ]:
# ── 1.2 Dtypes and memory ────────────────────────────────────────────────────
# Choosing the right dtype can save 4-8× memory for large fingerprint matrices

# Fingerprint bits are binary → uint8 (1 byte) not float64 (8 bytes)
n_compounds, n_bits = 10_000, 2048
fps_float = np.random.randint(0, 2, (n_compounds, n_bits), dtype=np.int32).astype(np.float64)
fps_uint8 = np.random.randint(0, 2, (n_compounds, n_bits), dtype=np.uint8)
fps_bool  = fps_uint8.astype(bool)

print("Fingerprint matrix memory comparison:")
print(f"  float64: {fps_float.nbytes / 1e6:.1f} MB")
print(f"  uint8:   {fps_uint8.nbytes  / 1e6:.1f} MB  ← 8× smaller")
print(f"  bool:    {fps_bool.nbytes   / 1e6:.1f} MB  ← 8× smaller")

# Common dtype choices in tox/chem
print("\nDtype guide:")
print("  Fingerprint bits  → bool or uint8")
print("  Descriptor values → float32 (ML input) or float64 (statistics)")
print("  Count data (RNA)  → int32 or int64")
print("  Class labels      → int8 or bool")
print("  p-values          → float64 (need full precision)")

# Practical: descriptor matrix for a library
descriptors = np.random.randn(1000, 200).astype(np.float32)  # float32 for ML
print(f"\nDescriptor matrix (float32): {descriptors.nbytes/1e6:.1f} MB")

---
## Section 2 — Indexing and Boolean Masking

In [ ]:
# ── 2.1 Boolean indexing — the workhorse for filtering compounds ───────────
# Simulate a descriptor matrix for 200 compounds
np.random.seed(42)
n_cpds = 200

mw   = np.random.normal(350, 100, n_cpds).clip(100, 900)
logp = np.random.normal(2.5, 1.5, n_cpds).clip(-3, 8)
tpsa = np.random.normal(80, 30, n_cpds).clip(0, 200)
hbd  = np.random.randint(0, 8, n_cpds)
hba  = np.random.randint(0, 12, n_cpds)
qed  = np.random.uniform(0.1, 0.9, n_cpds)

# Lipinski Ro5 filter — vectorised across all 200 compounds at once
ro5_mask = (mw <= 500) & (logp <= 5) & (hbd <= 5) & (hba <= 10)
print(f"Ro5 filter: {ro5_mask.sum()}/{n_cpds} compounds pass")

# Veber rules
veber_mask = (tpsa <= 140) & (np.random.randint(0, 15, n_cpds) <= 10)  # rotbonds sim.
print(f"Veber filter: {veber_mask.sum()}/{n_cpds} compounds pass")

# Combined filter
combined = ro5_mask & veber_mask & (qed >= 0.4)
print(f"Combined (Ro5 + Veber + QED≥0.4): {combined.sum()}/{n_cpds}")

# Apply mask to get passing compounds
mw_passing   = mw[combined]
logp_passing = logp[combined]
print(f"\nPassing compounds MW: mean={mw_passing.mean():.1f} ± {mw_passing.std():.1f}")
print(f"Passing compounds LogP: mean={logp_passing.mean():.2f} ± {logp_passing.std():.2f}")

In [ ]:
# ── 2.2 Fancy indexing and np.where ──────────────────────────────────────────
# Simulate 96-well plate viability data
np.random.seed(42)
plate = np.random.normal(100, 8, (8, 12))  # 8 rows × 12 columns

# Positive control (column 1) = 0% viability (full toxicity)
plate[:, 0] = np.random.normal(5, 3, 8)
# Negative control (column 12) = 100% viability
plate[:, 11] = np.random.normal(100, 5, 8)

print("96-well plate (% viability):")
print(np.round(plate, 1))

# Find wells with cytotoxicity (< 70% viability)
toxic_rows, toxic_cols = np.where(plate < 70)
print(f"\nCytotoxic wells (< 70%): {len(toxic_rows)}")
for r, c in zip(toxic_rows, toxic_cols):
    print(f"  Row {r+1}, Col {c+1}: {plate[r,c]:.1f}%")

# np.where: vectorised if-else
classification = np.where(plate >= 70, "viable", "toxic")
print(f"\nViable wells:  {(classification == 'viable').sum()}")
print(f"Toxic wells:   {(classification == 'toxic').sum()}")

# Normalise to percent of vehicle control (column 12)
vehicle_mean = plate[:, 11].mean()
normalised   = plate / vehicle_mean * 100
print(f"\nNormalised (% vehicle): min={normalised.min():.1f}  max={normalised.max():.1f}")

---
## Section 3 — Broadcasting

In [ ]:
# ── 3.1 Broadcasting for normalisation ───────────────────────────────────────
# Broadcasting: NumPy automatically expands dimensions to match shapes.
# This eliminates explicit loops over samples.

np.random.seed(42)
X = np.random.randn(500, 9).astype(np.float64)  # 500 compounds × 9 descriptors
# Columns: MW, LogP, TPSA, HBD, HBA, RotBonds, Rings, ArRings, QED

# Z-score normalisation — broadcasting handles [500,9] - [9] automatically
mu    = X.mean(axis=0)   # shape [9]  ← mean per descriptor
sigma = X.std(axis=0)    # shape [9]
Z = (X - mu) / sigma     # [500,9] - [9] → broadcasts to [500,9]

print(f"Input shape:      {X.shape}")
print(f"Mean per column:  {X.mean(axis=0).round(3)}")
print(f"After z-score, mean: {Z.mean(axis=0).round(6)}")   # ≈ 0
print(f"After z-score, std:  {Z.std(axis=0).round(6)}")    # ≈ 1

# Min-max normalisation to [0, 1]
xmin = X.min(axis=0)
xmax = X.max(axis=0)
X_mm = (X - xmin) / (xmax - xmin)    # broadcasting again
print(f"\nMin-max range: [{X_mm.min():.4f}, {X_mm.max():.4f}]")

# Pairwise Euclidean distance matrix — broadcasting trick
# ||xi - xj||² = ||xi||² + ||xj||² - 2 xi·xj
sq_norms = (X**2).sum(axis=1, keepdims=True)   # [500, 1]
D2 = sq_norms + sq_norms.T - 2 * X @ X.T        # [500, 500]
D  = np.sqrt(np.clip(D2, 0, None))
print(f"\nPairwise distance matrix: {D.shape}")
print(f"Mean nearest-neighbour distance: {np.sort(D, axis=1)[:,1].mean():.3f}")

---
## Section 4 — Linear Algebra for QSAR

In [ ]:
# ── 4.1 PCA from scratch with NumPy ──────────────────────────────────────────
# Understanding PCA at the matrix level is fundamental to QSAR analysis.

def pca_numpy(X: np.ndarray, n_components: int = 2) -> tuple:
    """
    PCA using eigen-decomposition of the covariance matrix.
    Steps:
      1. Centre the data (subtract mean)
      2. Compute covariance matrix C = X^T X / (n-1)
      3. Eigen-decompose: C = V Λ V^T
      4. Project: Z = X_centred @ V[:, :k]
    """
    # 1. Centre
    X_c = X - X.mean(axis=0)

    # 2. Covariance matrix
    C = X_c.T @ X_c / (X.shape[0] - 1)    # [n_features × n_features]

    # 3. Eigen-decomposition (sorted by eigenvalue, descending)
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues  = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # 4. Project
    Z = X_c @ eigenvectors[:, :n_components]

    var_explained = eigenvalues / eigenvalues.sum()
    return Z, eigenvectors, var_explained

# Simulated descriptor matrix (500 compounds × 9 descriptors)
np.random.seed(42)
n_feat = 9
X_desc = np.random.randn(300, n_feat)
# Add correlation structure (realistic)
X_desc[:, 2] = 0.7*X_desc[:, 0] + 0.3*X_desc[:, 2]   # MW correlated with rings
X_desc[:, 4] = 0.6*X_desc[:, 1] + 0.4*X_desc[:, 4]   # LogP correlated with HBA

Z, V, var_exp = pca_numpy(X_desc, n_components=2)
print(f"PCA: {X_desc.shape} → {Z.shape}")
print(f"Variance explained: PC1={var_exp[0]*100:.1f}%  PC2={var_exp[1]*100:.1f}%")
print(f"Cumulative (first 4): {var_exp[:4].cumsum()[-1]*100:.1f}%")

# Chemical interpretation: which descriptors load on PC1?
print("\nPC1 loadings (sorted by absolute value):")
desc_names = ["MW","LogP","TPSA","HBD","HBA","RotBonds","Rings","ArRings","QED"]
for name, load in sorted(zip(desc_names, V[:,0]), key=lambda x: abs(x[1]), reverse=True)[:5]:
    bar = "█" * int(abs(load)*30)
    print(f"  {name:10s}: {load:+.3f}  {bar}")

In [ ]:
# ── 4.2 Linear regression for QSAR ───────────────────────────────────────────
# Solve ordinary least squares: w = (X^T X)^{-1} X^T y

np.random.seed(42)
n = 200

# Simulate logS (aqueous solubility) as a function of descriptors
# True model: logS = -0.5*LogP + 0.01*(MW-300) - 0.02*TPSA + noise
logp_v = np.random.uniform(-1, 6, n)
mw_v   = np.random.uniform(150, 600, n)
tpsa_v = np.random.uniform(0, 150, n)
noise  = np.random.normal(0, 0.5, n)

logS_true = -0.5*logp_v + 0.01*(mw_v-300) - 0.02*tpsa_v
logS_obs  = logS_true + noise

# Design matrix X with intercept
X_reg = np.column_stack([np.ones(n), logp_v, mw_v, tpsa_v])  # [200, 4]

# OLS: w = (X^T X)^{-1} X^T y
XtX = X_reg.T @ X_reg           # [4, 4]
Xty = X_reg.T @ logS_obs        # [4]
w   = np.linalg.solve(XtX, Xty)  # [4] — more stable than np.linalg.inv

print("OLS QSAR model: logS ~ intercept + LogP + MW + TPSA")
print(f"  Intercept : {w[0]:+.3f}")
print(f"  LogP coef : {w[1]:+.3f}  (true: -0.500)")
print(f"  MW coef   : {w[2]:+.3f}  (true: +0.010)")
print(f"  TPSA coef : {w[3]:+.3f}  (true: -0.020)")

# Prediction and R²
logS_pred = X_reg @ w
ss_res = ((logS_obs - logS_pred)**2).sum()
ss_tot = ((logS_obs - logS_obs.mean())**2).sum()
r2 = 1 - ss_res/ss_tot
rmse = np.sqrt(ss_res/n)
print(f"\nR² = {r2:.4f}  RMSE = {rmse:.3f}")

---
## Section 5 — Statistics for Toxicology

In [ ]:
# ── 5.1 Descriptive statistics on assay data ─────────────────────────────────
np.random.seed(42)

# Simulate cell viability assay: 3 replicates × 8 concentrations
concs    = np.logspace(-3, 2, 8)   # 0.001 to 100 μM
# True Hill curve
ec50_true, n_true = 5.0, 1.5
viability_true = 100 / (1 + (concs/ec50_true)**n_true)

# Three experimental replicates
replicates = viability_true[:, None] + np.random.normal(0, 5, (8, 3))

# NumPy statistics functions
mean_rep  = replicates.mean(axis=1)
std_rep   = replicates.std(axis=1, ddof=1)   # ddof=1 for sample std
sem_rep   = std_rep / np.sqrt(3)
cv_rep    = std_rep / mean_rep * 100           # coefficient of variation

print(f"{'Conc (μM)':12s} {'Mean':>8} {'SD':>6} {'SEM':>6} {'CV%':>6}")
for c, m, s, e, cv in zip(concs, mean_rep, std_rep, sem_rep, cv_rep):
    print(f"{c:12.4f} {m:8.2f} {s:6.2f} {e:6.2f} {cv:6.1f}")

# Quality control metrics (Z'-factor for HTS)
pos_ctrl = np.random.normal(5,   3, 24)   # positive controls
neg_ctrl = np.random.normal(100, 5, 24)   # negative controls

mu_p, sd_p = pos_ctrl.mean(), pos_ctrl.std()
mu_n, sd_n = neg_ctrl.mean(), neg_ctrl.std()
zprime = 1 - 3*(sd_p + sd_n) / abs(mu_n - mu_p)

print(f"\nZ'-factor: {zprime:.3f}")
print(f"  (≥ 0.5 = excellent HTS assay quality, 0.5–1.0 is the target range)")

In [ ]:
# ── 5.2 Dose-response analysis with NumPy ────────────────────────────────────
# Hill equation: response = top / (1 + (EC50/C)^n)

def hill_4param(concs: np.ndarray,
                ec50: float, hill: float,
                top: float, bottom: float) -> np.ndarray:
    """4-parameter Hill equation."""    return bottom + (top - bottom) / (1 + (ec50/concs)**hill)

# Fit using scipy (standard practice)
from scipy.optimize import curve_fit

# Simulate hERG IC50 assay data
np.random.seed(42)
concs  = np.array([0.001, 0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0])
true_p = (1.5, 1.2, 95.0, 3.0)   # EC50, hill, top, bottom
y_obs  = hill_4param(concs, *true_p) + np.random.normal(0, 4, len(concs))

popt, pcov = curve_fit(
    hill_4param, concs, y_obs,
    p0=[1.0, 1.0, 100.0, 0.0],
    bounds=([1e-4, 0.1, 50.0, -10.0], [1000.0, 5.0, 120.0, 20.0]),
    maxfev=5000
)
perr = np.sqrt(np.diag(pcov))

ec50, hill, top, bot = popt
print(f"hERG IC50 Fit:")
print(f"  EC50  = {ec50:.3f} μM  ± {perr[0]:.3f}  (true: {true_p[0]})")
print(f"  Hill  = {hill:.3f}     ± {perr[1]:.3f}")
print(f"  Top   = {top:.1f}%     ± {perr[2]:.2f}")

# 95% confidence interval on EC50 using bootstrap
def bootstrap_ec50(concs, y_obs, n_boot=1000):
    """Estimate 95% CI on EC50 by bootstrap resampling."""    ec50s = []
    n = len(y_obs)
    for _ in range(n_boot):
        idx  = np.random.choice(n, n, replace=True)
        try:
            p, _ = curve_fit(hill_4param, concs[idx], y_obs[idx],
                              p0=popt, maxfev=500, bounds=([1e-4,0.1,50,-10],[1000,5,120,20]))
            ec50s.append(p[0])
        except RuntimeError:
            pass
    ec50s = np.array(ec50s)
    return np.percentile(ec50s, [2.5, 97.5])

ci = bootstrap_ec50(concs, y_obs, n_boot=500)
print(f"  EC50 95% CI: [{ci[0]:.3f}, {ci[1]:.3f}] μM (bootstrap, n=500)")

---
## Section 6 — Fingerprints at Scale

In [ ]:
# ── 6.1 Vectorised Tanimoto and related metrics ──────────────────────────────
def tanimoto_matrix(X: np.ndarray) -> np.ndarray:
    """Full N×N Tanimoto similarity matrix. X: binary [N, bits]."""    X   = X.astype(bool).astype(np.float32)
    ixj = X @ X.T                             # intersection
    sx  = X.sum(axis=1, keepdims=True)        # row bit counts
    union = sx + sx.T - ixj
    return np.where(union > 0, ixj/union, 0.0)

def cosine_matrix(X: np.ndarray) -> np.ndarray:
    """Cosine similarity matrix (useful for count fingerprints)."""    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_n   = X / np.where(norms > 0, norms, 1.0)
    return X_n @ X_n.T

def bulk_query(query_fp: np.ndarray, library_fps: np.ndarray) -> np.ndarray:
    """Fastest pattern: query (1D) vs library (2D), return similarity array."""    q   = query_fp.astype(bool).astype(np.float32)
    L   = library_fps.astype(bool).astype(np.float32)
    ixj = L @ q                               # [N_lib] intersections
    sq  = q.sum()
    sl  = L.sum(axis=1)                       # [N_lib] bit counts
    union = sq + sl - ixj
    return np.where(union > 0, ixj/union, 0.0)

# Simulate fingerprint library
np.random.seed(42)
N_LIB, N_BITS = 5000, 2048
library = (np.random.rand(N_LIB, N_BITS) > 0.85).astype(np.float32)
query   = (np.random.rand(N_BITS) > 0.85).astype(np.float32)

import time

# Bulk query: find nearest neighbours
t0   = time.perf_counter()
sims = bulk_query(query, library)
t1   = time.perf_counter()
print(f"Bulk query (5000 compounds, 2048 bits): {(t1-t0)*1000:.1f} ms")

# Top hits
top_idx = np.argsort(sims)[::-1][:5]
print(f"Top 5 hits: Tc = {sims[top_idx].round(4)}")

# Full similarity matrix (smaller library for demo)
t0  = time.perf_counter()
sim = tanimoto_matrix(library[:200])
t1  = time.perf_counter()
print(f"\n200×200 Tanimoto matrix: {(t1-t0)*1000:.1f} ms")
print(f"Mean pairwise Tc: {sim[np.triu_indices(200,1)].mean():.4f}")

# Diversity analysis using max-min (Kennard-Stone style)
# Find the compound LEAST similar to any current selection
selected   = [0]
candidates = list(range(1, 200))
for _ in range(9):
    max_min = -1
    best    = -1
    for c in candidates:
        min_sim = sim[c, selected].max()   # max Tc to any selected
        if min_sim > max_min:
            max_min, best = min_sim, c
    selected.append(best)
    candidates.remove(best)

print(f"\nDiverse subset (10 compounds): {selected}")

---
## Section 7 — Signal Processing for Assay Data

In [ ]:
# ── 7.1 MEA signal smoothing (moving average + Gaussian filter) ──────────────
# Multi-electrode array (MEA) records neural firing — raw signal is noisy.

from scipy.signal import savgol_filter, find_peaks
from scipy.ndimage import gaussian_filter1d

# Simulate neural spike train data from MEA
np.random.seed(42)
t        = np.linspace(0, 60, 6000)   # 60 seconds, 100 Hz sampling
baseline = np.random.normal(0, 0.5, len(t))
# Add neural burst events
spikes   = np.zeros_like(t)
for t_burst in [5, 12, 20, 30, 35, 45, 52]:
    n_spikes = np.random.randint(8, 20)
    for _ in range(n_spikes):
        idx = int((t_burst + np.random.uniform(-2, 2)) * 100)
        if 0 < idx < len(spikes):
            spikes[idx] += np.random.uniform(3, 8)   # spike amplitude

signal = baseline + spikes

# Smoothing methods
sma     = np.convolve(signal, np.ones(50)/50, mode='same')  # simple moving average
gauss   = gaussian_filter1d(signal, sigma=20)                # Gaussian
savgol  = savgol_filter(signal, window_length=51, polyorder=3)  # Savitzky-Golay

# Detect peaks (neural bursts)
peaks, props = find_peaks(savgol, height=1.5, distance=100, prominence=1.0)
print(f"Neural bursts detected: {len(peaks)}")
print(f"Mean inter-burst interval: {np.diff(t[peaks]).mean():.1f} s")
print(f"Mean burst amplitude: {savgol[peaks].mean():.2f}")

# Plot a section
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(t[:1000], signal[:1000], '#AAAAAA', lw=0.8, label='Raw MEA signal')
axes[0].plot(t[:1000], savgol[:1000], '#E74C3C', lw=2.0, label='Savitzky-Golay')
axes[0].set_ylabel('Amplitude (mV)'); axes[0].legend(fontsize=9)
axes[0].set_title('MEA Neural Signal — Smoothing Comparison', fontweight='bold')

axes[1].plot(t[:1000], signal[:1000], '#AAAAAA', lw=0.8, alpha=0.5)
axes[1].plot(t[:1000], sma[:1000],   '#1565C0', lw=1.8, label='Moving average')
axes[1].plot(t[:1000], gauss[:1000], '#27AE60', lw=1.8, label='Gaussian filter')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Amplitude (mV)')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## Section 8 — Monte Carlo & Bootstrapping

In [ ]:
# ── 8.1 Monte Carlo simulation in QSAR ───────────────────────────────────────
# Propagate uncertainty through a predictive model using MC sampling.

np.random.seed(42)

def predict_logS(mw: float, logp: float, tpsa: float) -> float:
    """Simplified Egan logS model."""    return -0.74*logp + 0.0091*mw - 0.0091*tpsa - 0.048

# A compound with measurement uncertainty
mw_meas, mw_std   = 320.0, 5.0     # MW ± 5 Da (measurement error)
logp_meas, logp_std = 2.5, 0.3     # LogP ± 0.3 (assay error)
tpsa_meas, tpsa_std = 75.0, 8.0    # TPSA ± 8 Å²

N_MC = 100_000
mw_samples   = np.random.normal(mw_meas,   mw_std,   N_MC)
logp_samples = np.random.normal(logp_meas, logp_std, N_MC)
tpsa_samples = np.random.normal(tpsa_meas, tpsa_std, N_MC)

logS_samples = predict_logS(mw_samples, logp_samples, tpsa_samples)

print(f"Monte Carlo uncertainty propagation (N={N_MC:,}):")
print(f"  logS = {logS_samples.mean():.3f} ± {logS_samples.std():.3f}  (1σ)")
print(f"  95% CI: [{np.percentile(logS_samples, 2.5):.3f}, "
      f"{np.percentile(logS_samples, 97.5):.3f}]")

# Bootstrap confidence interval on EC50
from scipy.optimize import curve_fit

def hill_2p(c, ec50, n):
    return 100 / (1 + (ec50/c)**n)

concs_sim = np.array([0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0])
y_sim = hill_2p(concs_sim, 5.0, 1.2) + np.random.normal(0, 4, 7)

ec50_boots = []
for _ in range(2000):
    idx = np.random.choice(len(y_sim), len(y_sim), replace=True)
    try:
        p, _ = curve_fit(hill_2p, concs_sim[idx], y_sim[idx],
                          p0=[5.0, 1.0], bounds=([0.001,0.1],[1000,5]), maxfev=300)
        ec50_boots.append(p[0])
    except RuntimeError:
        pass

ec50_boots = np.array(ec50_boots)
print(f"\nBootstrap EC50 (n=2000 resamples):")
print(f"  EC50 = {ec50_boots.mean():.3f} μM")
print(f"  95% CI: [{np.percentile(ec50_boots, 2.5):.3f}, "
      f"{np.percentile(ec50_boots, 97.5):.3f}] μM")

In [ ]:
# ── 8.2 NumPy performance tips ────────────────────────────────────────────────
import time

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                  NumPy Performance for Toxicology                       ║
╠══════════════════════════════════════════════════════════════════════════╣
║ VECTORISE EVERYTHING                                                     ║
║  ✗ for i in range(n): result[i] = fn(X[i])  ← Python loop, slow       ║
║  ✓ result = fn(X)                            ← vectorised, fast         ║
║  ✓ result = np.vectorize(fn)(X)              ← if fn not vectorisable  ║
╠══════════════════════════════════════════════════════════════════════════╣
║ CHOOSE RIGHT DTYPE                                                       ║
║  Fingerprints  → bool or uint8   (8× less memory than float64)         ║
║  ML input      → float32         (2× less memory, same precision)      ║
║  Statistics    → float64         (precision matters for p-values)      ║
║  RNA counts    → int32                                                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║ MEMORY-EFFICIENT OPERATIONS                                              ║
║  out= parameter: np.add(A, B, out=C)         avoids allocation          ║
║  in-place ops:  A += B                        no copy                   ║
║  Views not copies: A[mask] returns view if mask is slice               ║
╠══════════════════════════════════════════════════════════════════════════╣
║ FINGERPRINT OPERATIONS (2048-bit, 10K compounds)                        ║
║  Tanimoto matrix:  X @ X.T / (sum + sum.T - X @ X.T)  ← vectorised    ║
║  Bulk query:       sims = bulk_query(q, library)        ← see above     ║
║  Bit counting:     X.sum(axis=1)                        ← popcount      ║
╠══════════════════════════════════════════════════════════════════════════╣
║ DOSE-RESPONSE                                                            ║
║  EC50 fit:  scipy.optimize.curve_fit(hill, concs, y, p0=[...])         ║
║  EC50 CI:   bootstrap 1000-2000 resamples, np.percentile([2.5, 97.5])  ║
║  Z'-factor: 1 - 3*(sd_p + sd_n) / |mu_n - mu_p|   (> 0.5 = good)     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ STATISTICAL TESTS                                                        ║
║  Normality:        scipy.stats.shapiro(data)  (n < 50) or anderson     ║
║  Two groups:       ttest_ind (normal) / mannwhitneyu (non-parametric)  ║
║  Multiple groups:  f_oneway (ANOVA) / kruskal (non-parametric)         ║
║  Post-hoc:         pairwise_tukeyhsd (statsmodels)                     ║
║  FDR correction:   multipletests(pvals, method='fdr_bh')               ║
╚══════════════════════════════════════════════════════════════════════════╝
""")